In [34]:
# this notebook will normalize the data
# while keeping key level as feature
import pandas as pd
import json
from pathlib import Path


folder_path = Path.cwd().parents[1] / "data" / "mlData"
src_path = folder_path / "BTCUSD-15m-raw.jsonl"

# Read JSONL file - keep timestamp as raw number
df = pd.read_json(src_path, lines=True, convert_dates=False)
df.head()

,timestamp,open,high,low,close,vol,isSTR,isITR,isLTR
0,1708659900000,51099.929688,51120.000000,51052.199219,51100.000000,209.838379,0,0,0
1,1708660800000,51100.000000,51106.871094,51003.000000,51091.859375,230.937881,0,0,0
2,1708661700000,51091.871094,51236.640625,51031.101562,51236.640625,333.208618,0,0,0
3,1708662600000,51236.628906,51321.921875,51183.210938,51224.031250,284.106201,0,0,0
4,1708663500000,51224.039062,51254.199219,51195.839844,51219.589844,169.352463,0,0,0


In [35]:
def add_keylv(df: pd.DataFrame) -> pd.DataFrame:
    n = len(df)
    ltr_keylv = [None] * n
    itr_keylv = [None] * n
    bars_since_ltr = [0] * n
    bars_since_itr = [0] * n

    ltr_keylv[0] = df.iloc[0]["low"]
    itr_keylv[0] = df.iloc[0]["low"]

    for i in range(1, n):
        ltr_sig = df.iloc[i]["isLTR"]
        itr_sig = df.iloc[i]["isITR"]

        if ltr_sig == 1:
            ltr_keylv[i] = round(df.iloc[i]["high"], 4)
            bars_since_ltr[i] = 0
        elif ltr_sig == -1:
            ltr_keylv[i] = round(df.iloc[i]["low"], 4)
            bars_since_ltr[i] = 0
        else:
            ltr_keylv[i] = ltr_keylv[i - 1]
            bars_since_ltr[i] = bars_since_ltr[i - 1] + 1

        if itr_sig == 1:
            itr_keylv[i] = round(df.iloc[i]["high"], 4)
            bars_since_itr[i] = 0
        elif itr_sig == -1:
            itr_keylv[i] = round(df.iloc[i]["low"], 4)
            bars_since_itr[i] = 0
        else:
            itr_keylv[i] = itr_keylv[i - 1]
            bars_since_itr[i] = bars_since_itr[i - 1] + 1

    df = df.copy()
    df["ITRkeylv"] = itr_keylv
    df["barsSinceITR"] = bars_since_itr
    df["LTRkeylv"] = ltr_keylv
    df["barsSinceLTR"] = bars_since_ltr
    return df


df = add_keylv(df)
df.tail()


,timestamp,open,high,low,close,vol,isSTR,isITR,isLTR,ITRkeylv,barsSinceITR,LTRkeylv,barsSinceLTR
70925,1772492400000,69440.476562,69479.578125,69215.398438,69223.242188,121.096069,0,0,0,70096.0,25,65056.0,97
70926,1772493300000,69223.250000,69261.687500,68990.976562,69002.929688,223.747665,0,0,0,70096.0,26,65056.0,98
70927,1772494200000,69002.921875,69014.257812,68758.398438,68796.859375,195.868637,0,0,0,70096.0,27,65056.0,99
70928,1772495100000,68796.859375,68965.671875,68740.000000,68830.062500,97.017410,0,0,0,70096.0,28,65056.0,100
70929,1772496000000,68830.062500,69004.398438,68750.679688,68809.992188,168.970734,0,0,0,70096.0,29,65056.0,101


In [36]:
# create column
df["hiDTK_LTR"]    = ((df["high"]  - df["LTRkeylv"]) / df["LTRkeylv"]).round(6)
df["lowDTK_LTR"]   = ((df["low"]   - df["LTRkeylv"]) / df["LTRkeylv"]).round(6)
df["closeDTK_LTR"] = ((df["close"] - df["LTRkeylv"]) / df["LTRkeylv"]).round(6)

df["hiDTK_ITR"]    = ((df["high"]  - df["ITRkeylv"]) / df["ITRkeylv"]).round(6)
df["lowDTK_ITR"]   = ((df["low"]   - df["ITRkeylv"]) / df["ITRkeylv"]).round(6)
df["closeDTK_ITR"] = ((df["close"] - df["ITRkeylv"]) / df["ITRkeylv"]).round(6)

# create Y then shift it by 1
diff = df["close"] - df["open"]
df["Y"] = diff.apply(lambda x: 1 if x > 0 else (-1 if x < 0 else 0))
df["Y"] = df["Y"].shift(-1)  # shift down by 1, row[-1] becomes NaN
df.tail()


,timestamp,open,high,low,close,vol,isSTR,isITR,isLTR,ITRkeylv,barsSinceITR,LTRkeylv,barsSinceLTR,hiDTK_LTR,lowDTK_LTR,closeDTK_LTR,hiDTK_ITR,lowDTK_ITR,closeDTK_ITR,Y
70925,1772492400000,69440.476562,69479.578125,69215.398438,69223.242188,121.096069,0,0,0,70096.0,25,65056.0,97,0.067996,0.063936,0.064056,-0.008794,-0.012563,-0.012451,-1.0
70926,1772493300000,69223.250000,69261.687500,68990.976562,69002.929688,223.747665,0,0,0,70096.0,26,65056.0,98,0.064647,0.060486,0.060670,-0.011902,-0.015764,-0.015594,-1.0
70927,1772494200000,69002.921875,69014.257812,68758.398438,68796.859375,195.868637,0,0,0,70096.0,27,65056.0,99,0.060844,0.056911,0.057502,-0.015432,-0.019082,-0.018534,1.0
70928,1772495100000,68796.859375,68965.671875,68740.000000,68830.062500,97.017410,0,0,0,70096.0,28,65056.0,100,0.060097,0.056628,0.058013,-0.016125,-0.019345,-0.018060,-1.0
70929,1772496000000,68830.062500,69004.398438,68750.679688,68809.992188,168.970734,0,0,0,70096.0,29,65056.0,101,0.060692,0.056792,0.057704,-0.015573,-0.019193,-0.018346,NaN


# Prepare for training data


In [37]:
# cleaned data : ready for input
df = df.drop(columns=["open", "high", "low", "close", "vol", "LTRkeylv", "ITRkeylv"])
df.tail()

,timestamp,isSTR,isITR,isLTR,barsSinceITR,barsSinceLTR,hiDTK_LTR,lowDTK_LTR,closeDTK_LTR,hiDTK_ITR,lowDTK_ITR,closeDTK_ITR,Y
70925,1772492400000,0,0,0,25,97,0.067996,0.063936,0.064056,-0.008794,-0.012563,-0.012451,-1.0
70926,1772493300000,0,0,0,26,98,0.064647,0.060486,0.060670,-0.011902,-0.015764,-0.015594,-1.0
70927,1772494200000,0,0,0,27,99,0.060844,0.056911,0.057502,-0.015432,-0.019082,-0.018534,1.0
70928,1772495100000,0,0,0,28,100,0.060097,0.056628,0.058013,-0.016125,-0.019345,-0.018060,-1.0
70929,1772496000000,0,0,0,29,101,0.060692,0.056792,0.057704,-0.015573,-0.019193,-0.018346,NaN


In [38]:
# save to JSONL for training
out_path = folder_path / "BTCUSD-15m-input.jsonl"

# note drop that NaN Y at the last row
df.dropna(subset=["Y"]).to_json(out_path, orient="records", lines=True)
print(f"Saved {len(df.dropna(subset=['Y']))} rows to {out_path}")

Saved 70929 rows to /Users/aimliu/Library/CloudStorage/OneDrive-Personal/_CODE/Python/SMC-bot/data/mlData/BTCUSD-15m-input.jsonl
